# Exercise 5.1 — Evaluating a regression model on training and test data

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-01-regression-metrics.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

On Day 3 you fitted a straight line to predict glucose and measured its error
with a loss function. A model is only useful if it also predicts **people it has
not seen**. In this exercise we fit a model on the training rows, predict both
the training and the test rows, and compare the errors. The comparison tells us
whether the model is **overfitting** (much better on training than on test) or
**underfitting** (similar on both, but poor on both).

Suppose we believe that the relationship between glucose and age is not a
straight line but a **cubic**:

$$\widehat{\text{glucose}} = \beta_0 + \beta_1\,\text{age} + \beta_2\,\text{age}^2 + \beta_3\,\text{age}^3$$

**Theory:** polynomial features as a pre-processing step; recipes, workflows
and pipelines; predictions on training and test data; mean squared error, root
mean squared error and mean absolute error; overfitting and underfitting.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

The data are in [`data/diabetes_clean.csv`](data/diabetes_clean.csv). We use the same 750
people and the same split as on Day 4: every fifth person is held back as the
test set.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["glucose", "mass", "age", "diabetes"]]

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

print(diabetes.shape)
print(train.shape)
print(test.shape)

750 people: 600 for training (80%) and 150 for testing (20%).

------------------------------------------------------------------------

### Part 1 — A cubic model as a pre-processing step

To fit a cubic we do not need a new kind of model. We create new columns from
age (its square and its cube) and fit an ordinary linear regression on them.
Creating new columns from old ones is **feature engineering**, so in
scikit-learn it is a step of the **pipeline**, alongside the other pre-processing
steps you met on Day 2. `PolynomialFeatures()` adds the polynomial columns:

In [ ]:
poly_step = PolynomialFeatures(degree=3, include_bias=False)   # age, age^2, age^3

Look at what the step does to the data. `fit_transform()` works out the
transformation from the training rows and applies it to them:

In [ ]:
cubic_features = poly_step.fit_transform(train[["age"]])

pd.DataFrame(cubic_features, columns=poly_step.get_feature_names_out()).head(10)

The three columns are age, age² and age³. The squared and cubed values are
large (81³ = 531441), which is fine for a cubic.

Now put the polynomial step and the model together in a **pipeline** and fit it
on the training rows:

In [ ]:
cubic_pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("model", LinearRegression()),
])
cubic_pipe = cubic_pipe.fit(train[["age"]], train["glucose"])

print("intercept:   ", round(cubic_pipe["model"].intercept_, 3))
print("coefficients:", cubic_pipe["model"].coef_)

The coefficients belong to age, age² and age³. They differ from the ones you get
in R, where `step_poly()` uses orthogonal polynomials, but the **predictions**
are identical in both languages, and the predictions are what we evaluate.

Here is the fitted cubic on top of the training data:

In [ ]:
age_grid = pd.DataFrame({"age": np.arange(21, 82)})
curve = cubic_pipe.predict(age_grid)

fig, ax = plt.subplots()
ax.scatter(train["age"], train["glucose"], alpha=0.3, color="black")
ax.plot(age_grid["age"], curve, color="red", linewidth=2)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title("Cubic fit of glucose on age, training rows")
plt.show()

> **Learning points**
>
> - A polynomial model is a linear regression on new columns made from the predictor.
> - Making those columns is pre-processing, so it lives in the recipe (R) or the pipeline (Python), together with the model.
> - R's `step_poly()` uses orthogonal polynomials and Python's `PolynomialFeatures()` uses raw powers: different coefficients, identical predictions.

------------------------------------------------------------------------

### Part 2 — Predictions on the training and the test rows

We want to know whether the model **generalises**: does it predict new people as
well as the people it was fitted on? So we predict both sets. `predict()` gives
an array of predictions:

In [ ]:
cubic_pipe.predict(train[["age"]])[:6].round(2)

It is easier to keep the predictions next to the data. Copy each data set and
add the predictions as a new column `pred`, for the training rows and the test
rows:

In [ ]:
train_preds = train.copy()
train_preds["pred"] = cubic_pipe.predict(train[["age"]])

test_preds = test.copy()
test_preds["pred"] = cubic_pipe.predict(test[["age"]])

test_preds.head().round(2)

The first test person is 28 years old with a glucose of 194, and the model
predicts 127.4 for them: an error of 66.6 mg/dL, a person the model gets badly
wrong.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, data, name in [(axes[0], train_preds, "train (600)"), (axes[1], test_preds, "test (150)")]:
    data = data.sort_values("age")
    ax.scatter(data["age"], data["glucose"], alpha=0.3, color="black")
    ax.plot(data["age"], data["pred"], color="red", linewidth=2)
    ax.set_title(name)
    ax.set_xlabel("Age (years)")
axes[0].set_ylabel("Glucose (mg/dL)")
fig.suptitle("The same fitted cubic, on the training and the test rows")
plt.show()

> **Learning points**
>
> - The model is fitted on the training rows only; the test rows are used for predicting, never for fitting.
> - Keep the predictions next to the data: `augment()` in R, a new column in Python.

------------------------------------------------------------------------

### Part 3 — Measuring the error, by hand and with a package

For a regression model the error of one person is $y_i - \hat{y}_i$, the observed
glucose minus the predicted glucose. Three summaries of these errors are used
everywhere; you met them as loss functions on Day 3:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 \qquad
\text{RMSE} = \sqrt{\text{MSE}} \qquad
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

RMSE and MAE are in the units of the outcome, mg/dL, so they can be read as
"the typical size of an error". MSE is in squared units. Compute all three on the
training rows, one step at a time:

In [ ]:
train_error = train_preds["glucose"] - train_preds["pred"]

train_mse  = np.mean(train_error ** 2)
train_rmse = np.sqrt(train_mse)
train_mae  = np.mean(np.abs(train_error))

print("MSE: ", round(train_mse, 2))
print("RMSE:", round(train_rmse, 2))
print("MAE: ", round(train_mae, 2))

On the training rows the typical error is about 30 mg/dL (RMSE 30.29, MAE
24.61). Now do the same for the **test** rows:

In [ ]:
# Replace every ______ below, then run this cell.
test_error = ______

test_mse  = np.mean(test_error ** 2)
test_rmse = ______
test_mae  = np.mean(np.abs(test_error))

print("MSE: ", round(test_mse, 2))
print("RMSE:", round(test_rmse, 2))
print("MAE: ", round(test_mae, 2))

> **Stuck?**
>
> Copy the training cell and change `train_preds` to `test_preds`. The error is
> the observed glucose minus the prediction `pred`, and RMSE is the square root
> of MSE.

You will compute these metrics for every model you build, so packages already
do it. In scikit-learn they are in `sklearn.metrics`: each takes the truth
first and the prediction second.

In [ ]:
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error

print("train MSE: ", round(mean_squared_error(train_preds["glucose"], train_preds["pred"]), 2))
print("train RMSE:", round(root_mean_squared_error(train_preds["glucose"], train_preds["pred"]), 2))
print("train MAE: ", round(mean_absolute_error(train_preds["glucose"], train_preds["pred"]), 2))

The same numbers as by hand. Check the test rows with the package as well:

In [ ]:
# Replace every ______ below, then run this cell.
print("test MSE: ", round(mean_squared_error(______, ______), 2))
print("test RMSE:", round(root_mean_squared_error(test_preds["glucose"], test_preds["pred"]), 2))
print("test MAE: ", round(mean_absolute_error(test_preds["glucose"], test_preds["pred"]), 2))

**Question!** The model is about as good on the test rows as on the training
rows. Is it overfitting? Is it a good model? A typical error of 30 mg/dL, when
most glucose values lie between 80 and 180, is large. What would you try next?

The errors are similar on both sets, so there is no sign of **overfitting**. But
they are large on both, which is **underfitting**: age alone cannot capture what
drives a person's glucose, whatever the shape of the curve. The natural next step
is to add predictors, not to make the curve more flexible.

> **Learning points**
>
> - MSE, RMSE and MAE summarise the errors $y_i - \hat{y}_i$; RMSE and MAE are in the units of the outcome.
> - Always compute them on both the training and the test rows, and compare.
> - Much better on training than on test: overfitting. Similar but poor on both: underfitting.

------------------------------------------------------------------------

### Part 4 — More flexible is not more accurate

Was the cubic the right choice? Fit polynomials of degree 1 (a straight line) up
to 8, and record the training and test RMSE of each. The code is the same as in
Part 1 with a different `degree`, repeated in a loop. One change: age is scaled
first with `StandardScaler()`. Age to the power 8 has fifteen digits, too many
for the computer's arithmetic to stay accurate; scaling first keeps the numbers
small and gives the same predictions.

In [ ]:
from sklearn.preprocessing import StandardScaler

rows = []
for d in range(1, 9):
    poly_pipe = Pipeline([
        ("scale", StandardScaler()),
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("model", LinearRegression()),
    ])
    poly_pipe = poly_pipe.fit(train[["age"]], train["glucose"])

    train_rmse = root_mean_squared_error(train["glucose"], poly_pipe.predict(train[["age"]]))
    test_rmse  = root_mean_squared_error(test["glucose"], poly_pipe.predict(test[["age"]]))
    rows.append({"degree": d, "train_rmse": train_rmse, "test_rmse": test_rmse})

degree_results = pd.DataFrame(rows)
degree_results.round(3)

In [ ]:
fig, ax = plt.subplots()
ax.plot(degree_results["degree"], degree_results["train_rmse"], marker="o", label="train_rmse")
ax.plot(degree_results["degree"], degree_results["test_rmse"], marker="o", label="test_rmse")
ax.set_xlabel("Degree of the polynomial")
ax.set_ylabel("RMSE (mg/dL)")
ax.set_title("Training error keeps falling; test error does not")
ax.legend()
plt.show()

The training RMSE falls with every extra degree, from 31.10 for the straight line
to 30.12 for degree 8: a more flexible curve can always follow the training
points more closely. The test RMSE does not follow. It is lowest at degree 2
(29.08) and highest at degree 8 (29.74). The extra flexibility is spent on
the noise of these particular 600 people, which does not repeat in new people.
With 600 people and a single predictor the effect is small; with fewer people
or more flexible models it can be very large.

**Question!** Which degree would you choose, and on which data did you base the
choice? (Look again at the Stretch below.)

> **Learning points**
>
> - Training error always improves as a model becomes more flexible, so it cannot be used to choose how flexible a model should be.
> - Test error stops improving, and starts to get worse, once the model fits noise: that is overfitting.

## Stretch

> **Stretch — choosing with the test set spends the test set**
>
> Part 4 chose a degree by looking at the test RMSE, and the test RMSE of that
> degree is now an optimistic estimate of how it will do on new people. How
> could you choose the degree using the training rows only, and keep the test
> rows for one final check? Try it: split the 600 training rows again into 480
> and 120, choose the degree on the 120, then report the test RMSE once.